<a href="https://colab.research.google.com/github/mloulaps-rgb/---/blob/main/HCI_%D0%9B%D0%A04_%D0%9C%D0%B0%D1%88%D0%B8%D0%BD%D0%BD%D0%B8%D0%B9_%D0%BF%D0%B5%D1%80%D0%B5%D0%BA%D0%BB%D0%B0%D0%B4_LSH_%D0%92%D0%B5%D0%BB%D0%B8%D1%87%D0%BA%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Лабораторна робота №4. Наївний машинний переклад та локально-сенситивне хешування.
# Крок 1. Завантаження векторів слів (fastText) та словників перекладів (MUSE).
# Мови за варіантом: мова-1 = українська (uk), мова-2 = англійська (en).
!pip install -q gensim

import numpy as np
import re
import string
import nltk
from gensim.models import KeyedVectors
from nltk.corpus import twitter_samples, stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer
nltk.download('twitter_samples')
nltk.download('stopwords')

# Потокове завантаження лише перших 100000 векторів кожної мови:
# curl качає архів, zcat розпаковує "на льоту", sed бере перші 100000 рядків
# і завершується (q) -> завантажується лише потрібна частина, а не весь файл.
!curl -sL https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.uk.300.vec.gz | zcat | sed -n '2,100001p;100001q' > uk_body.txt
!curl -sL https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz | zcat | sed -n '2,100001p;100001q' > en_body.txt
# додаємо коректний заголовок word2vec-формату "кількість розмірність"
!printf '100000 300\n' | cat - uk_body.txt > cc.uk.300.vec
!printf '100000 300\n' | cat - en_body.txt > cc.en.300.vec

# словники перекладів MUSE (uk -> en): 0-5000 на навчання, 5000-6500 на тест
!wget -q -O l1-l2.train.txt https://dl.fbaipublicfiles.com/arrival/dictionaries/uk-en.0-5000.txt
!wget -q -O l1-l2.test.txt  https://dl.fbaipublicfiles.com/arrival/dictionaries/uk-en.5000-6500.txt
!ls -lh cc.uk.300.vec cc.en.300.vec l1-l2.train.txt l1-l2.test.txt

[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Unzipping corpora/twitter_samples.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


-rw-r--r-- 1 root root 216M Nov 15 20:50 cc.en.300.vec
-rw-r--r-- 1 root root 217M Nov 15 20:50 cc.uk.300.vec
-rw-r--r-- 1 root root  45K Jan 31  2019 l1-l2.test.txt
-rw-r--r-- 1 root root 164K Jan 31  2019 l1-l2.train.txt


In [2]:
# Крок 1 (продовження). Завантаження векторів слів у пам'ять.
lang1_embeddings = KeyedVectors.load_word2vec_format("cc.uk.300.vec", binary=False)  # українська
lang2_embeddings = KeyedVectors.load_word2vec_format("cc.en.300.vec", binary=False)  # англійська

print("Слів у словнику мови-1 (uk):", len(lang1_embeddings.index_to_key))
print("Слів у словнику мови-2 (en):", len(lang2_embeddings.index_to_key))
print("Розмірність векторів:", lang1_embeddings.vector_size)

Слів у словнику мови-1 (uk): 100000
Слів у словнику мови-2 (en): 100000
Розмірність векторів: 300


In [3]:
# Крок 2. Завантаження двомовних словників перекладів.
def get_dict(file_name):
    """Зчитує словник перекладів: кожен рядок - пара 'слово_мови1 слово_мови2'."""
    etof = {}
    with open(file_name, encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 2:
                etof[parts[0]] = parts[1]
    return etof

l1_l2_train = get_dict("l1-l2.train.txt")
l1_l2_test  = get_dict("l1-l2.test.txt")
print("Розмір словника перекладів для тренування:", len(l1_l2_train))
print("Розмір словника перекладів для тестування:", len(l1_l2_test))
# приклад кількох пар
for w1 in list(l1_l2_train)[:5]:
    print(f"  {w1} -> {l1_l2_train[w1]}")

Розмір словника перекладів для тренування: 4999
Розмір словника перекладів для тестування: 1500
  категорія -> category
  від -> by
  для -> for
  також -> also
  про -> about


In [4]:
# Крок 3. Побудова матриць векторів слів X та Y + функція косинусної подібності.
def cosine_similarity(A, B):
    """Косинусна подібність. A може бути вектором або матрицею, B - вектор."""
    dot = np.dot(A, B)
    normb = np.linalg.norm(B)
    if len(A.shape) == 1:
        norma = np.linalg.norm(A)
        cos = dot / (norma * normb)
    else:
        norma = np.linalg.norm(A, axis=1)
        epsilon = 1.0e-9
        cos = dot / (norma * normb + epsilon)
    return cos

def get_matrices(l1_l2, l1_embeddings, l2_embeddings):
    """Будує матриці X (вектори слів мови-1) та Y (вектори перекладів мовою-2)."""
    X_l, Y_l = [], []
    l1_set = set(l1_embeddings.index_to_key)
    l2_set = set(l2_embeddings.index_to_key)
    for l1_word, l2_word in l1_l2.items():
        if l1_word in l1_set and l2_word in l2_set:
            X_l.append(l1_embeddings[l1_word])
            Y_l.append(l2_embeddings[l2_word])
    return np.asarray(X_l), np.asarray(Y_l)

X_train, Y_train = get_matrices(l1_l2_train, lang1_embeddings, lang2_embeddings)
print("Форма матриці X_train:", X_train.shape)
print("Форма матриці Y_train:", Y_train.shape)

Форма матриці X_train: (4189, 300)
Форма матриці Y_train: (4189, 300)


In [8]:
# Крок 4. Пошук матриці перетворення R методом градієнтного спуску.
def compute_loss(X, Y, R):
    """Функція втрат: середньоквадратична норма Фробеніуса різниці XR - Y."""
    m = X.shape[0]
    diff = np.dot(X, R) - Y
    return np.sum(diff ** 2) / m

def compute_gradient(X, Y, R):
    """Градієнт функції втрат за R."""
    m = X.shape[0]
    return np.dot(X.T, np.dot(X, R) - Y) * (2 / m)

def align_embeddings(X, Y, train_steps=2000, learning_rate=0.8, verbose=True):
    np.random.seed(129)
    n = X.shape[1]
    R = np.random.rand(n, n)
    for i in range(train_steps):
        if verbose and i % 200 == 0:
            print(f"Функція втрат на ітерації {i}: {compute_loss(X, Y, R):.4f}")
        R -= learning_rate * compute_gradient(X, Y, R)
    return R

R_train = align_embeddings(X_train, Y_train, train_steps=2000, learning_rate=0.8)
print("\\nМатрицю перетворення R навчено. Форма:", R_train.shape)

Функція втрат на ітерації 0: 89.1543
Функція втрат на ітерації 200: 13.8300
Функція втрат на ітерації 400: 5.4408
Функція втрат на ітерації 600: 2.8775
Функція втрат на ітерації 800: 1.8355
Функція втрат на ітерації 1000: 1.3413
Функція втрат на ітерації 1200: 1.0830
Функція втрат на ітерації 1400: 0.9382
Функція втрат на ітерації 1600: 0.8525
Функція втрат на ітерації 1800: 0.7994
\nМатрицю перетворення R навчено. Форма: (300, 300)


In [9]:
# Кроки 5-6. Функція перекладу слова та оцінка точності на тестовому наборі.
def translate(word, R, l1_embeddings, l2_embeddings, k=1):
    """Переклад слова мови-1 на мову-2 через перетворення R і пошук
    найближчого вектора мови-2 за косинусною подібністю."""
    if word not in l1_embeddings.key_to_index:
        return "Слово відсутнє у словнику"
    l1_emb = l1_embeddings[word]
    l2_emb = np.dot(l1_emb, R)                      # перетворений вектор
    sims = cosine_similarity(l2_embeddings.vectors, l2_emb)
    top_k = sims.argsort()[-k:][::-1]
    return l2_embeddings.index_to_key[top_k[0]]

def test_vocabulary(test_file, R, l1_embeddings, l2_embeddings):
    """Точність перекладу (P@1). Одному слову мови-1 у словнику MUSE може
    відповідати кілька правильних перекладів - переклад вважаємо правильним,
    якщо він збігається з будь-яким із них."""
    refs = {}
    with open(test_file, encoding="utf-8") as f:
        for line in f:
            p = line.split()
            if len(p) >= 2:
                refs.setdefault(p[0], set()).add(p[1])
    correct, total = 0, 0
    for l1_word, l2_set in refs.items():
        if l1_word not in l1_embeddings.key_to_index:
            continue
        if not any(w in l2_embeddings.key_to_index for w in l2_set):
            continue
        total += 1
        if translate(l1_word, R, l1_embeddings, l2_embeddings, k=1) in l2_set:
            correct += 1
    return (correct / total if total else 0.0), correct, total

# приклади перекладу
for w in ["кіт", "місто", "вода", "книга", "день", "сонце", "мова"]:
    print(f"{w}  ->  {translate(w, R_train, lang1_embeddings, lang2_embeddings)}")

acc, correct, total = test_vocabulary("l1-l2.test.txt", R_train, lang1_embeddings, lang2_embeddings)
print(f"\\nТочність перекладу на тестовому наборі (P@1): {acc:.3f} ({acc*100:.2f}%)  [{correct}/{total}]")

кіт  ->  cat
місто  ->  city
вода  ->  water
книга  ->  book
день  ->  day
сонце  ->  sun
мова  ->  language
\nТочність перекладу на тестовому наборі (P@1): 0.443 (44.26%)  [524/1184]


In [10]:
# Кроки 7-8. Завантаження твітів та подання кожного твіта вектором ознак.
all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')
all_tweets = all_positive_tweets + all_negative_tweets

def process_tweet(tweet):
    """Очищення та токенізація твіта (без стемінгу - для кращого збігу зі словником)."""
    tweet = re.sub(r'\$\w*', '', tweet)
    tweet = re.sub(r'^RT[\s]+', '', tweet)
    tweet = re.sub(r'https?://[^\s\n\r]+', '', tweet)
    tweet = re.sub(r'#', '', tweet)
    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
    tokens = tokenizer.tokenize(tweet)
    stopwords_english = stopwords.words('english')
    return [w for w in tokens if w not in stopwords_english and w not in string.punctuation]

def get_tweet_embedding(tweet, embeddings):
    """Вектор твіта = середнє векторів його слів."""
    vecs = [embeddings[w] for w in process_tweet(tweet) if w in embeddings.key_to_index]
    if not vecs:
        return np.zeros(embeddings.vector_size)
    return np.mean(vecs, axis=0)

def get_all_tweet_embeddings(tweets, embeddings):
    return {i: get_tweet_embedding(t, embeddings) for i, t in enumerate(tweets)}

# твіти англійською -> використовуємо англійські вектори (мова-2)
tweet_embeddings = get_all_tweet_embeddings(all_tweets, lang2_embeddings)
print("Кількість твітів:", len(tweet_embeddings))
print("Розмірність векторів твітів:", tweet_embeddings[0].shape)

Кількість твітів: 10000
Розмірність векторів твітів: (300,)


In [11]:
# Кроки 9-10. Локально-сенситивне хешування (LSH): хеш, таблиці, пошук сусідів.
def hash_func(embedding, planes):
    """Хеш вектора: біт 1/0 за знаком скалярного добутку на кожну гіперплощину."""
    hash_value = 0
    for i in range(planes.shape[1]):
        sign = 1 if np.dot(embedding, planes[:, i]) > 0 else 0
        hash_value += sign * (2 ** i)
    return hash_value

def make_hash_table(embeddings, planes):
    """Хеш-таблиця {хеш: [індекси векторів]}."""
    table = {}
    for i, emb in enumerate(embeddings):
        table.setdefault(hash_func(emb, planes), []).append(i)
    return table

def init_lsh(embeddings, n_planes, n_tables):
    """Генерує n_tables наборів по n_planes випадкових гіперплощин і будує таблиці."""
    np.random.seed(0)
    planes_list = [np.random.normal(size=(embeddings.shape[1], n_planes)) for _ in range(n_tables)]
    tables = [make_hash_table(embeddings, planes) for planes in planes_list]
    return tables, planes_list

def lsh_knn(tweet_id, embedding, tables, planes_list, tweet_embeddings, k=5):
    """Пошук k найближчих твітів за допомогою LSH (кандидати з корзин)."""
    candidates = set()
    for table, planes in zip(tables, planes_list):
        candidates.update(table.get(hash_func(embedding, planes), []))
    candidates.discard(tweet_id)
    candidates = list(candidates)
    cand_emb = np.array([tweet_embeddings[i] for i in candidates])
    sims = cosine_similarity(cand_emb, embedding)
    top = np.argsort(sims)[-k:][::-1]
    return [candidates[i] for i in top]

emb_matrix = np.array([tweet_embeddings[i] for i in range(len(tweet_embeddings))])
tables, planes_list = init_lsh(emb_matrix, n_planes=10, n_tables=10)
print("Побудовано хеш-таблиць:", len(tables))
print("Корзин у першій таблиці:", len(tables[0]))

Побудовано хеш-таблиць: 10
Корзин у першій таблиці: 579


In [12]:
# Крок 11. Тестування пошуку подібних твітів.
for tweet_id in [0, 2000]:
    tweet = all_tweets[tweet_id]
    print("=" * 70)
    print(f"Запит (твіт #{tweet_id}): {tweet}")
    neighbors = lsh_knn(tweet_id, tweet_embeddings[tweet_id], tables, planes_list,
                        tweet_embeddings, k=3)
    print("Найбільш схожі твіти:")
    for nid in neighbors:
        print(f"  [{nid}] {all_tweets[nid]}")
    print()

Запит (твіт #0): #FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)
Найбільш схожі твіти:
  [51] #FollowFriday @France_Espana @reglisse_menthe @CCI_inter for being top engaged members in my community this week :)
  [195] #FollowFriday @CCIdeLYON @CCI_95 @CPNERI_ESIGROUP for being top engaged members in my community this week :)
  [253] #FollowFriday @CCIFCcanada @AdamEvnmnt @boxcalf1 for being top engaged members in my community this week :)

Запит (твіт #2000): @jayyycgee Well you're about to get recruited to Team Doom. So The Nasty Crew is going to be affiliated. &gt;:)
Найбільш схожі твіти:
  [1153] @ankitsri24 Sure thing :) We have shared your suggestion with the concerned team, we are sure to get them back pretty soon.
  [2109] left @TheColtRising wasn't a good fit for the team, goodluck in the future guys! :)
  [899] @morriskid are you coming to Bergerac?  Have the first team left any u21's I'd know of? #checking for ch